# LSTM Forecasting Baseline (PyTorch, Kaggle-Ready)

This notebook contains the full LSTM pipeline code directly in cells.

What it does:
- Loads preprocessed splits (`train.parquet`, `val.parquet`, `test.parquet`)
- Builds leakage-safe time-series sequences per `(store_nbr, family)`
- Trains an LSTM with PyTorch
- Evaluates with MAE, RMSE, RMSLE
- Compares against baseline RMSLE references
- Saves trained model artifact

It is designed to run both:
- locally from `data/processed/`
- on Kaggle from `/kaggle/input/...` parquet files

In [ ]:
from dataclasses import dataclass
from pathlib import Path
from typing import Dict, List, Tuple
import glob
import os

import numpy as np
import pandas as pd

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error

SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", DEVICE)
print("PyTorch:", torch.__version__)

In [ ]:
@dataclass
class LSTMConfig:
    sequence_length: int = 30
    batch_size: int = 256
    hidden_size: int = 128
    num_layers: int = 2
    dropout: float = 0.2
    learning_rate: float = 1e-3
    epochs: int = 7


def rmsle(y_true: np.ndarray, y_pred: np.ndarray) -> float:
    y_true = np.clip(y_true, a_min=0.0, a_max=None)
    y_pred = np.clip(y_pred, a_min=0.0, a_max=None)
    return float(np.sqrt(np.mean((np.log1p(y_pred) - np.log1p(y_true)) ** 2)))


def find_processed_paths() -> Tuple[Path, Path, Path]:
    local_dir = Path("data") / "processed"
    if (local_dir / "train.parquet").exists() and (local_dir / "val.parquet").exists() and (local_dir / "test.parquet").exists():
        return local_dir / "train.parquet", local_dir / "val.parquet", local_dir / "test.parquet"

    # Kaggle fallback: search common input mount recursively
    candidates = glob.glob("/kaggle/input/**/train.parquet", recursive=True)
    for train_file in candidates:
        train_path = Path(train_file)
        base = train_path.parent
        val_path = base / "val.parquet"
        test_path = base / "test.parquet"
        if val_path.exists() and test_path.exists():
            return train_path, val_path, test_path

    raise FileNotFoundError(
        "Could not find train/val/test parquet files. "
        "Expected in data/processed/ or /kaggle/input/**/."
    )


def create_sequences(
    df: pd.DataFrame,
    feature_cols: List[str],
    sequence_length: int,
    target_split: str,
    target_col: str = "sales",
    group_cols: List[str] = ["store_nbr", "family"],
) -> Tuple[np.ndarray, np.ndarray]:
    # Enforce deterministic order for sequence generation.
    df = df.sort_values(group_cols + ["date"]).reset_index(drop=True)

    X_list: List[np.ndarray] = []
    y_list: List[float] = []

    for _, grp in df.groupby(group_cols, sort=False):
        grp = grp.reset_index(drop=True)
        feat = grp[feature_cols].to_numpy(dtype=np.float32)
        target = grp[target_col].to_numpy(dtype=np.float32)
        split = grp["split"].to_numpy()

        for i in range(sequence_length, len(grp)):
            # Target row must belong to the requested split.
            if split[i] != target_split:
                continue
            # Uses only past timesteps [i-seq, ..., i-1].
            X_list.append(feat[i - sequence_length : i])
            y_list.append(float(target[i]))

    if not X_list:
        raise ValueError(f"No sequences created for split: {target_split}")

    X = np.stack(X_list).astype(np.float32)
    y = np.array(y_list, dtype=np.float32)
    return X, y


class SalesSequenceDataset(Dataset):
    def __init__(self, X: np.ndarray, y: np.ndarray) -> None:
        self.X = torch.tensor(X, dtype=torch.float32)
        self.y = torch.tensor(y, dtype=torch.float32)

    def __len__(self) -> int:
        return len(self.y)

    def __getitem__(self, idx: int):
        return self.X[idx], self.y[idx]


class LSTMModel(nn.Module):
    def __init__(self, input_size: int, hidden_size: int = 128, num_layers: int = 2, dropout: float = 0.2):
        super().__init__()
        self.lstm = nn.LSTM(
            input_size=input_size,
            hidden_size=hidden_size,
            num_layers=num_layers,
            batch_first=True,
            dropout=dropout if num_layers > 1 else 0.0,
        )
        self.fc = nn.Linear(hidden_size, 1)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        out, _ = self.lstm(x)
        last_hidden = out[:, -1, :]
        pred = self.fc(last_hidden)
        return pred.squeeze(-1)


def train_lstm(model: nn.Module, train_loader: DataLoader, val_loader: DataLoader, epochs: int, lr: float, device: torch.device):
    criterion = nn.MSELoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    model.to(device)

    for epoch in range(1, epochs + 1):
        model.train()
        train_losses = []
        for X_batch, y_batch in train_loader:
            X_batch, y_batch = X_batch.to(device), y_batch.to(device)

            optimizer.zero_grad()
            preds = model(X_batch)
            loss = criterion(preds, y_batch)
            loss.backward()
            optimizer.step()
            train_losses.append(float(loss.item()))

        model.eval()
        val_losses = []
        with torch.no_grad():
            for X_batch, y_batch in val_loader:
                X_batch, y_batch = X_batch.to(device), y_batch.to(device)
                preds = model(X_batch)
                loss = criterion(preds, y_batch)
                val_losses.append(float(loss.item()))

        print(
            f"Epoch {epoch:02d}/{epochs} | "
            f"Train MSE: {np.mean(train_losses):.5f} | "
            f"Val MSE: {np.mean(val_losses):.5f}"
        )


def predict(model: nn.Module, loader: DataLoader, device: torch.device) -> Tuple[np.ndarray, np.ndarray]:
    model.eval()
    y_true_all, y_pred_all = [], []
    with torch.no_grad():
        for X_batch, y_batch in loader:
            X_batch = X_batch.to(device)
            preds = model(X_batch).cpu().numpy()
            y_pred_all.append(preds)
            y_true_all.append(y_batch.numpy())

    y_true = np.concatenate(y_true_all)
    y_pred = np.concatenate(y_pred_all)
    y_pred = np.clip(y_pred, a_min=0.0, a_max=None)
    return y_true, y_pred


def evaluate_predictions(y_true: np.ndarray, y_pred: np.ndarray) -> Dict[str, float]:
    return {
        "MAE": float(mean_absolute_error(y_true, y_pred)),
        "RMSE": float(np.sqrt(mean_squared_error(y_true, y_pred))),
        "RMSLE": float(rmsle(y_true, y_pred)),
    )

In [ ]:
config = LSTMConfig(
    sequence_length=30,
    batch_size=256,
    hidden_size=128,
    num_layers=2,
    dropout=0.2,
    learning_rate=1e-3,
    epochs=7,
)

train_path, val_path, test_path = find_processed_paths()
print("train:", train_path)
print("val  :", val_path)
print("test :", test_path)

train_df = pd.read_parquet(train_path)
val_df = pd.read_parquet(val_path)
test_df = pd.read_parquet(test_path)

for split_name, split_df in (("train", train_df), ("val", val_df), ("test", test_df)):
    split_df["date"] = pd.to_datetime(split_df["date"])
    split_df["split"] = split_name

# Concatenate to allow val/test to use earlier context windows (still no future leakage).
full_df = pd.concat([train_df, val_df, test_df], ignore_index=True)
full_df = full_df.sort_values(["store_nbr", "family", "date"]).reset_index(drop=True)

# Encode family if it is text.
if full_df["family"].dtype == object:
    family_vocab = {name: idx for idx, name in enumerate(sorted(train_df["family"].astype(str).unique()))}
    for df in (train_df, val_df, test_df, full_df):
        df["family"] = df["family"].astype(str).map(family_vocab).fillna(-1).astype(np.int16)

TARGET_COL = "sales"
EXCLUDE_COLS = {"date", TARGET_COL, "split"}
feature_cols = [c for c in full_df.columns if c not in EXCLUDE_COLS]

# Fit scaler on train split only to avoid leakage.
scaler = StandardScaler()
scaler.fit(full_df.loc[full_df["split"] == "train", feature_cols])
full_df.loc[:, feature_cols] = scaler.transform(full_df[feature_cols]).astype(np.float32)
full_df.loc[:, TARGET_COL] = full_df[TARGET_COL].astype(np.float32)

X_train, y_train = create_sequences(full_df, feature_cols, config.sequence_length, "train", target_col=TARGET_COL)
X_val, y_val = create_sequences(full_df, feature_cols, config.sequence_length, "val", target_col=TARGET_COL)
X_test, y_test = create_sequences(full_df, feature_cols, config.sequence_length, "test", target_col=TARGET_COL)

train_ds = SalesSequenceDataset(X_train, y_train)
val_ds = SalesSequenceDataset(X_val, y_val)
test_ds = SalesSequenceDataset(X_test, y_test)

train_loader = DataLoader(train_ds, batch_size=config.batch_size, shuffle=True, drop_last=False)
val_loader = DataLoader(val_ds, batch_size=config.batch_size, shuffle=False, drop_last=False)
test_loader = DataLoader(test_ds, batch_size=config.batch_size, shuffle=False, drop_last=False)

print("Feature count:", len(feature_cols))
print("X_train shape:", X_train.shape, "y_train shape:", y_train.shape)
print("X_val shape  :", X_val.shape, "y_val shape  :", y_val.shape)
print("X_test shape :", X_test.shape, "y_test shape :", y_test.shape)

In [ ]:
model = LSTMModel(
    input_size=len(feature_cols),
    hidden_size=config.hidden_size,
    num_layers=config.num_layers,
    dropout=config.dropout,
)

train_lstm(
    model=model,
    train_loader=train_loader,
    val_loader=val_loader,
    epochs=config.epochs,
    lr=config.learning_rate,
    device=DEVICE,
)

val_true, val_pred = predict(model, val_loader, DEVICE)
test_true, test_pred = predict(model, test_loader, DEVICE)

val_metrics = evaluate_predictions(val_true, val_pred)
test_metrics = evaluate_predictions(test_true, test_pred)

print("Validation metrics:", val_metrics)
print("Test metrics      :", test_metrics)

# Baseline references from your baseline notebook.
baselines = pd.DataFrame([
    {"model": "naive", "validation_rmsle": 3.920209, "test_rmsle": 3.859030},
    {"model": "ridge", "validation_rmsle": 2.051011, "test_rmsle": 2.022613},
    {"model": "random_forest", "validation_rmsle": 1.208159, "test_rmsle": 1.214093},
    {"model": "lstm", "validation_rmsle": val_metrics["RMSLE"], "test_rmsle": test_metrics["RMSLE"]},
]).sort_values("validation_rmsle")

baselines

In [ ]:
# Save model artifact (Kaggle-compatible path)
output_dir = Path("/kaggle/working") if Path("/kaggle/working").exists() else Path("artifacts")
output_dir.mkdir(parents=True, exist_ok=True)

model_path = output_dir / "lstm_baseline.pt"
torch.save(model.state_dict(), model_path)
print("Saved model to:", model_path)

# Optional: save metrics
metrics_df = pd.DataFrame([
    {"split": "validation", **val_metrics},
    {"split": "test", **test_metrics},
])
metrics_path = output_dir / "lstm_metrics.csv"
metrics_df.to_csv(metrics_path, index=False)
print("Saved metrics to:", metrics_path)
metrics_df